# MOTEL Workflow v2 Demo

This notebook demonstrates the full Workflow v2 vertical slice:

1. Setup and bootstrap demo records.
2. Generate mapping suggestions from catalog matching.
3. Approve one suggestion.
4. Run duplicate detection.

## What you should see

- Unmapped and mapped record counts at each step.
- Suggested technology/process IDs with confidence.
- Real record files written under dev/workflow_data.

## Where files are stored

- Unmapped records: dev/workflow_data/unmapped_records
- Mapped records: dev/workflow_data/mapped_records
- Suggestions: dev/workflow_data/suggestions/latest.yaml

In [7]:
from pathlib import Path
import sys

# Ensure repository root is importable when notebook starts in dev/.
repo_root = Path.cwd().resolve().parents[0]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dev.workflow_v2.settings import WorkflowPaths
from dev.workflow_v2.pipeline import WorkflowEngine
from dev.workflow_v2.bootstrap import bootstrap_demo_records

# Build workflow paths and engine, then seed demo data if folders are empty.
paths = WorkflowPaths.from_repo_root(repo_root)
engine = WorkflowEngine(paths)
bootstrap_demo_records()

# Show queue state: (unmapped_count, mapped_count)
len(engine.list_unmapped()), len(engine.list_mapped())

(2, 0)

## Guideline for (unmapped) records
Use this structure for every new **unmapped** record before it is reviewed and approved into the techbase.

### 1) Purpose

An unmapped record should capture the user submission in a clean, reviewable format, **without final ontology mapping**.

### 2) Required fields

- `temp_id`: Temporary ID from intake (example: `TMP_0003`)
- `title`: Short, human-readable name
- `description`: Clear description of the technology/process
- `source_name`: Where the information came from
- `source_reference`: URL, document ID, or citation
- `submitted_by`: User or system that created the record
- `submitted_at`: ISO timestamp (`YYYY-MM-DDTHH:MM:SSZ`)

### 3) Optional but recommended fields

- `aliases`: Alternative names or synonyms
- `keywords`: Terms that improve matching quality
- `notes`: Extra reviewer context
- `region`: Geographic relevance
- `sector`: Domain/industry tag

### 4) Do **not** include in unmapped records

- Final `technology_id`
- Final `process_id`
- Reviewer approval metadata  
These are added only after suggestion review/approval.

### 5) YAML template

```yaml
tech_nam: "Rooftop solar PV (residential)"
tech_description: "Grid-connected residential photovoltaic installation."
source_name: "User submission form"
source_description: "the user submissions form for a research project"
source_reference: "https://example.org/form/123"

other_names:
    - "Home solar"
    - "Residential PV"
keywords:
    - solar
    - photovoltaic
    - rooftop

scope:
    - geographic_scope: "EU"
    - temporal_scope: "Buildings"
    - capacity_scope: "residenal, less than 50 kW"
    - system_boundary: "all installation and setup for the PV system at home"

    - notes: "User requested review for small-scale systems only."
```

### 6) Quality checklist

- Description is specific and unambiguous.
- Source is traceable.
- Spelling and units are consistent.
- No final mapping IDs are pre-filled.
- Content is complete enough for suggestion generation.

## Add a new record (user input style)

The next cell creates a brand-new record directly from user-provided values.

Notes:
- You do not provide `temp_id`; it is generated automatically after submission.
- The output intentionally shows only essential fields.

In [13]:
# Create a new record from user-style inputs (minimal required fields).
description_input = "Residential rooftop solar PV system in Switzerland with typical capex assumptions for 2026."
source_id_input = "SRC_USER_001"
attribute_id_input = "ATTR_CAPEX"
value_input = 920.0
unit_input = "CHF/kW"

scope_input = {
    "geographic_scope": "GEO_CH",
    "temporal_scope": "TIME_2026",
    "capacity_scope": "CAP_RESIDENTIAL",
    "system_boundary": "COND_GRID_CONNECTED",
}

new_record = engine.create_and_submit(
    description=description_input,
    scope=scope_input,
    source_id=source_id_input,
    attribute_id=attribute_id_input,
    value=value_input,
    unit=unit_input,
)

# Show essential fields only (no temp_id, no internal timestamps/metadata).
essential_view = {
    "record_id": new_record.record_id,
    "status": new_record.status,
    "description": new_record.description,
    "scope": new_record.scope.__dict__,
    "sources": [{"source_id": s.source_id} for s in new_record.sources],
    "values": [
        {
            "attribute_id": v.attribute_id,
            "value": v.value,
            "unit": v.unit,
            "value_type": v.value_type,
        }
        for v in new_record.values
    ],
}

essential_view

{'record_id': 'REC_0003',
 'status': 'unmapped',
 'description': 'Residential rooftop solar PV system in Switzerland with typical capex assumptions for 2026.',
 'scope': {'geographic_scope': 'GEO_CH',
  'temporal_scope': 'TIME_2026',
  'capacity_scope': 'CAP_RESIDENTIAL',
  'system_boundary': 'COND_GRID_CONNECTED'},
 'sources': [{'source_id': 'SRC_USER_001'}],
 'values': [{'attribute_id': 'ATTR_CAPEX',
   'value': 920.0,
   'unit': 'CHF/kW',
   'value_type': 'numeric'}]}

In [9]:
# Generate mapping suggestions for every unmapped record.
suggestions = engine.build_suggestions()

# Inspect suggestion payloads (tech/process IDs, confidence, rationale).
[s.to_dict() for s in suggestions]

[{'record_id': 'REC_0001',
  'temp_id': 'TMP_0001',
  'suggested_tech_id': 'T_C_Conv_Solar_PV',
  'suggested_process_id': 'P_Electricity_Generation',
  'confidence': 0.3497,
  'method': 'rule_fuzzy',
  'rationale': 'Top name similarity match: Solar photovoltaic system',
  'ontology_iri': 'motel:T_C_Conv_Solar_PV'},
 {'record_id': 'REC_0002',
  'temp_id': 'TMP_0002',
  'suggested_tech_id': 'T_S_Stor_Battery',
  'suggested_process_id': 'P_Charging',
  'confidence': 0.386,
  'method': 'rule_fuzzy',
  'rationale': 'Top name similarity match: Battery energy storage',
  'ontology_iri': 'motel:T_S_Stor_Battery'}]

## Approve one suggestion and inspect mapped output

The next cell takes the first suggestion, approves it, and moves the record from unmapped to mapped.

After approval, we will print the raw mapped record file so you can see added mapping metadata (tech/process/confidence/reviewer).

In [10]:
# Approve the first suggestion and move the record into mapped_records.
if suggestions:
    first = suggestions[0].record_id
    mapped_path = engine.approve_suggestion(
        first,
        reviewer="notebook_user",
        notes="Approved in notebook demo",
    )
    print(mapped_path)

# Show queue state after approval: (unmapped_count, mapped_count)
len(engine.list_unmapped()), len(engine.list_mapped())

E:\Barton\repositories\motel-platform\dev\workflow_data\mapped_records\REC_0001.yaml


(1, 1)

In [11]:
# Print the raw mapped record file to inspect mapping metadata.
mapped_files = sorted(paths.mapped_dir.glob("REC_*.yaml"))
print("Mapped files:", [p.name for p in mapped_files])

if mapped_files:
    latest_mapped_path = mapped_files[-1]
    print("\n--- Raw mapped record ---")
    print(latest_mapped_path.name)
    print(latest_mapped_path.read_text(encoding="utf-8"))
else:
    print("No mapped records found.")

Mapped files: ['REC_0001.yaml']

--- Raw mapped record ---
REC_0001.yaml
{
  "record_id": "REC_0001",
  "temp_id": "TMP_0001",
  "description": "Rooftop solar photovoltaic system for residential electricity generation in Switzerland with typical capex assumptions.",
  "status": "mapped",
  "scope": {
    "geographic_scope": "GEO_CH",
    "temporal_scope": "TIME_2026",
    "capacity_scope": "CAP_RESIDENTIAL",
    "system_boundary": "COND_GRID_CONNECTED"
  },
  "sources": [
    {
      "source_id": "SRC_NREL_ATB_2023",
      "relevance": "primary"
    }
  ],
  "assumptions": [],
  "values": [
    {
      "attribute_id": "ATTR_CAPEX",
      "value": 980.0,
      "value_type": "numeric",
      "unit": "CHF/kW",
      "uncertainty": null,
      "note": null
    }
  ],
  "mapping": {
    "mapped_tech_id": "T_C_Conv_Solar_PV",
    "mapped_process_id": "P_Electricity_Generation",
    "confidence": 0.3497,
    "method": "rule_fuzzy",
    "mapping_notes": "Approved in notebook demo",
    "ontolo

## Duplicate detection

The next cell checks for potential duplicate records across unmapped and mapped sets.

An empty list means no duplicate pairs were detected under current matching rules.

In [12]:
engine.dedupe_report_as_dict()

[]